# Plot Juno's trajectory.

In [ ]:
import pandas as pd
import datetime as dt
import numpy as np 
import glob
import matplotlib.pyplot as plt 

from matplotlib.ticker import AutoMinorLocator

from JunoPy.spice import InitSpiceKernels, get_juno_position
from JunoPy.mag import MagData

## Get the spacecraft position for the given times.

In [ ]:
start = dt.datetime(2016, 6, 20, 17, 30)
end   = dt.datetime(2020, 7, 27, 16)

time_array = pd.date_range(
        start=start, end=end, periods=10000).to_pydatetime().tolist()

InitSpiceKernels()
position, _ = get_juno_position(time_array)

In [ ]:
def split_position(position):
    """Given the position of the spacecraft, split the array for points insdie 10 deg cone."""
    latitude = np.abs(np.arctan2(position[:, 2], np.sqrt(position[:, 0]**2 + position[:, 1]**2))) * 180 / np.pi
    print(latitude)
    position_masked = np.ma.empty(position.shape)
    position_masked[:, 0] = np.ma.masked_where(latitude > 10, position[:, 0])
    position_masked[:, 1] = np.ma.masked_where(latitude > 10, position[:, 1])
    position_masked[:, 2] = np.ma.masked_where(latitude > 10, position[:, 2])
    
    print(np.ma.count_masked(position_masked))
    return position_masked
    
position_masked = split_position(position)

## Calculate the Joy et al. (2002) magnetopause and bow-shock model

In [ ]:
def calc_coeff_bs(p):
    A = -1.107 + 1.591 * p**(-0.25)
    B = -0.566 - 0.812 * p**(-0.25)
    C =  0.048 - 0.059 * p**(-0.25)
    D =  0.077 - 0.038 * p
    E = -0.874 - 0.299 * p
    F = -0.055 + 0.124 * p
    return A, B, C, D, E, F
    
def calc_coeff_mp(p):
    A = -0.134 + 0.488 * p**(-0.25)
    B = -0.581 - 0.225 * p**(-0.25)
    C = -0.186 - 0.016 * p**(-0.25)
    D = -0.014 + 0.096 * p
    E = -0.814 - 0.811 * p
    F = -0.050 + 0.168 * p
    return A, B, C, D, E, F

def calc_equatorial(p, N, ctype='MP'):
    
    if 'MP' in ctype:
        A, B, C, D, E, F = calc_coeff_mp(p)
    else:
        A, B, C, D, E, F = calc_coeff_bs(p)
        
    phi = np.linspace(-150, 150, N) * np.pi / 180
    
    a = C * np.cos(phi)**2 + E * np.sin(phi)**2 + F * np.sin(phi) * np.cos(phi)
    b = B * np.cos(phi) + D * np.sin(phi)
    c = A
    
    root1 = (-b + np.sqrt(b**2 - 4 * a * c)) / (2 * a)
    root2 = (-b - np.sqrt(b**2 - 4 * a * c)) / (2 * a)
    
    r = np.zeros(N)
    for i in range(N):
        if root1[i] > 0:
            r[i] = root1[i]
        else:
            r[i] = root2[i]
        
    x = r * 120 * np.cos(phi)
    y = r * 120 * np.sin(phi)    
    

    return x, y
    

x_MP, y_MP = calc_equatorial(0.383, 101, ctype='MP')
x_BS, y_BS = calc_equatorial(0.382, 101, ctype='BS')

## Read the MVA epochs file and get the spacecraft position for the corresponding time

In [ ]:
def read_csv_file(plasmoidtype):
    start_times = []
    end_times = []
    durations = []
    with open('auto_plasmoid_list_20210329_window10.csv', 'r') as f:
        f.readline()
        for line in f:
            if plasmoidtype in line:
                start_times.append(dt.datetime.strptime(line.split(',')[0].strip(), '%Y-%jT%H:%M:%S'))
                end_times.append(dt.datetime.strptime(line.split(',')[1].strip(), '%Y-%jT%H:%M:%S'))
                durations.append((end_times[-1] - start_times[-1]).total_seconds())
    return start_times, end_times, durations

## Redo the MVA analysis for each interval and obtain the eigenvectors. 

In [ ]:
def read_and_crop(magfile, start, end):
    """Read and crop magnetometer data."""
    magdata = MagData(filename=magfile)
    start_cut = start - dt.timedelta(minutes=10)
    end_cut   = end   + dt.timedelta(minutes=10)
    magdata.df = magdata.df.loc[start_cut:end_cut]

    magdata_mva = MagData(filename=magfile)
    magdata_mva.df = magdata_mva.df.loc[start:end].copy()
    magdata_mva.perform_mva()

    return magdata, magdata_mva

In [ ]:
def get_mva_eigenvectors(starttimes, endttimes):

    x1 = []
    x2 = []
    x3 = []
    for start, end in zip(starttimes, endtimes):

        magfile = glob.glob('data/mag/*{:s}ss*.hdf5'.format(start.strftime('%Y%j')))[0]
        magdata, magdata_mva = read_and_crop(magfile, start, end)
        x1.append(magdata_mva.x1)
        x2.append(magdata_mva.x2)
        x3.append(magdata_mva.x3)

    x1 = np.asarray(x1)
    x2 = np.asarray(x2)
    x3 = np.asarray(x3)

    for i in range(len(x3)):
        if x3[i, 2] < 0:
            x1[i, :] = x1[i, :] * -1
            x2[i, :] = x2[i, :] * -1
            x3[i, :] = x3[i, :] * -1
    
    return x1, x2, x3

## Plot the trajectory

In [ ]:
def plot_trajectory_eigenvectors(
    position, x_BS, y_BS, x_MP, y_MP, position_mva, durations, plasmoidtype=None, dosave=False):

    # fig = plt.figure(figsize=(8, 4), sharex=True, constrained_layout=False, dpi=600)
    gs_kw = dict(height_ratios=[150, 75], hspace=0)
    fig, ax = plt.subplots(2, 1, sharex=True, dpi=600, gridspec_kw=gs_kw, figsize=(6, 8))

    ax1 = ax[0]
    ax2 = ax[1]

    ax1.plot(position[:, 0], position[:, 1], color=[0.3, 0.3, 0.3], zorder=1, alpha=0.5)
    ax2.plot(position[:, 0], position[:, 2], color=[0.3, 0.3, 0.3], zorder=1, alpha=0.5)
    # ax1.plot(position_masked[:, 0], position_masked[:, 1], color=[0.3, 0.3, 0.3], zorder=1)
    # ax2.plot(position_masked[:, 0], position_masked[:, 2], color=[0.3, 0.3, 0.3], zorder=1)

    ax1.plot(x_BS, y_BS, color='k', zorder=0)
    ax1.plot(x_MP, y_MP, color='k', ls='dashed', zorder=0)

    ax1.xaxis.set_minor_locator(AutoMinorLocator())
    ax1.yaxis.set_minor_locator(AutoMinorLocator())
    ax2.xaxis.set_minor_locator(AutoMinorLocator())
    ax2.yaxis.set_minor_locator(AutoMinorLocator())
    ax1.grid('on', which='minor', lw=0.5, ls='dotted')
    ax1.grid(color='k', ls='dashed', lw=0.25)
    ax2.grid('on', which='minor', lw=0.5, ls='dotted')
    ax2.grid(color='k', ls='dashed', lw=0.25)

    ax1.set_xlabel('X [R]', weight='bold')
    ax1.set_ylabel('Y [R]', weight='bold')
    ax2.set_xlabel('X [R]', weight='bold')
    ax2.set_ylabel('Z [R]', weight='bold')



    # ax1.set_xlim([-105, 40])
    ax1.set_ylim([-120, 30])

    ax2.set_xlim([-105, 40])
    ax2.set_ylim([-55, 20])

    # ax1.set_adjustable("datalim", share=True)

    ax1.set_aspect('equal', share=True)
    ax2.set_aspect('equal', share=True)
    
    sc1 = ax1.scatter(position_mva[:, 0], position_mva[:, 1], c=durations, zorder=2, marker='o', cmap='jet', vmin=0, vmax=200)    
    sc2 = ax2.scatter(position_mva[:, 0], position_mva[:, 2], c=durations, zorder=2, marker='o', cmap='jet', vmin=0, vmax=200)
    
    if plasmoidtype == 'flux-rope':
        q1 = ax1.quiver(position_mva[:, 0], position_mva[:, 1], x2[:, 0], x2[:, 1])
        ax1.text(-100, 20, 'Vectors of $x_M$', weight='bold')
    
    elif plasmoidtype == 'o-line':
        q1 = ax1.quiver(position_mva[:, 0], position_mva[:, 1], x1[:, 0], x1[:, 1])
        ax1.text(-100, 20, 'Vectors of $x_L$', weight='bold')
        

    q2 = ax2.quiver(position_mva[:, 0], position_mva[:, 2], x3[:, 0], x3[:, 2])
    ax2.text(-100, 10, 'Vectors of $x_N$', weight='bold')
    
    bbox2 = ax2.get_position()
    bbox1 = ax1.get_position()
    cax = fig.add_axes([bbox2.x1, bbox2.y0, 0.016, bbox1.y1- bbox2.y0])


    ax1.arrow(20, -115, 10, 0, color='k', head_width=3)
    ax1.text(20, -110, 'To Sun')
    ax1.text(30, -60, 'MP')
    ax1.text(30, -90, 'BS')
    
    ax2.text(-60, -40, 'Juno Trajectory', color=[0.3, 0.3, 0.3], alpha=0.5, weight='bold')

    ax1.add_artist(plt.Circle((0, 0), 1.0, color='k', zorder=0))
    ax2.add_artist(plt.Circle((0, 0), 1.0, color='k', zorder=0))

    cbar2 = fig.colorbar(sc2, cax=cax);
    cbar2.set_label('Peak-to-peak Duration (s)')
    plt.subplots_adjust(wspace=0.3)
    ax[0].set_title('Plasmoid type - {:s}, N={:d}'.format(plasmoidtype, len(durations)))
    
    if dosave:
        fig.savefig('out_images/TrajectoryLocationofEvent_quiver_{:s}.png'.format(plasmoidtype))


## For flux ropes

In [ ]:
starttimes, endtimes, durations = read_csv_file('flux-rope')

position_mva_fluxrope, _ = get_juno_position(starttimes)

x1, x2, x3 = get_mva_eigenvectors(starttimes, endtimes)

plot_trajectory_eigenvectors(
    position, x_BS, y_BS, x_MP, y_MP, position_mva_fluxrope, durations, plasmoidtype='flux-rope', dosave=True)
for time in starttimes:
    print(time.strftime('%Y-%j T %H:%M:%S'))
    
x2_fluxrope = x2 * 1.0

## For O-lines


In [ ]:
starttimes, endtimes, durations = read_csv_file('o-line')

position_mva_olines, _ = get_juno_position(starttimes)

x1, x2, x3 = get_mva_eigenvectors(starttimes, endtimes)

plot_trajectory_eigenvectors(
    position, x_BS, y_BS, x_MP, y_MP, position_mva_olines, durations, plasmoidtype='o-line', dosave=True)

x1_oline = x1 * 1.0

In [ ]:
def plot_dphi(position_mva_fluxrope, position_mva_oline, x2_fluxrope, x1_oline):
    
    
    rot = np.array([[0, -1, 0], [1, 0, 0], [0, 0, 1]])
    
    # Flux rope
    
    position_mva_fluxrope[:, 2] = 0
    azimuthal_direction = np.matmul(rot, position_mva_fluxrope.T).T
    for row in azimuthal_direction:
        row /= np.sqrt(np.sum(row**2))
    
    dphi = np.arccos(np.sum(np.multiply(azimuthal_direction, x2_fluxrope), axis=1)) * 180 / np.pi
    # dphi[dphi > 90] = dphi[dphi > 90] - 180
    
    dphi_fr = dphi * 1.0
    
    # O lines
    
    position_mva_oline[:, 2] = 0
    azimuthal_direction = np.matmul(rot, position_mva_oline.T).T
    for row in azimuthal_direction:
        row /= np.sqrt(np.sum(row**2))
    
    dphi = np.arccos(np.sum(np.multiply(azimuthal_direction, x1_oline), axis=1)) * 180 / np.pi
    # dphi[dphi > 90] = dphi[dphi > 90] - 180
    
    dphi_oline = dphi * 1.0
    
    fig, ax = plt.subplots(1, 2, dpi=150, gridspec_kw=dict(wspace=0.3))
    ax[0].hist(dphi_fr, bins=np.linspace(0, 180, 9))
    ax[1].hist(dphi_oline, bins=np.linspace(0, 180, 9))
    
    ax[0].set_title('a) Flux ropes, N={:}'.format(len(dphi_fr)))
    ax[1].set_title('b) O-lines, N={:}'.format(len(dphi_oline)))
    
    ax[0].set_xlabel('$\Delta \Phi (\mathbf{x}_M)$ ($^\circ$)', weight='bold')
    ax[1].set_xlabel('$\Delta \Phi (\mathbf{x}_L)$ ($^\circ$)', weight='bold')
    for a in ax:
        a.set_ylabel('Occurence #', weight='bold')
        a.set_xlim([0, 180])
        a.set_xticks([0, 45, 90, 135, 180])
    
    fig.savefig('out_images/histogram_dphi.png', bbox_inches='tight' )
    plt.show()
    
    
    
plot_dphi(position_mva_fluxrope, position_mva_olines, x2_fluxrope, x1_oline)